# Learning Objectives

* Build a fan-out / fan-in Pegasus workflow from shell wrappers and an Apptainer container.
* Limit concurrent jobs of one kind (SRA downloads) with a DAGMan category.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the merged alignment results produced by a run.

# SRA Search Workflow

This workflow downloads paired-end sequencing runs from the NCBI
[Sequence Read Archive](https://www.ncbi.nlm.nih.gov/sra) and aligns each one against a reference genome
(the crAssphage genome in `examples/`). Every SRA ID is downloaded and aligned independently, in parallel;
the resulting BAM files are merged into a single tarball through a tree of merge jobs (at most 25 inputs per
merge). See `README.md` in this repository for the full description.

**Workflow Jobs:**
1. `bowtie2-build` - index the reference FASTA once
2. `fasterq-dump` - download one SRA run's paired FASTQ files (at most 20 at a time)
3. `bowtie2` - align the reads, sort and index the BAM (`samtools`)
4. `merge` - merge BAM/BAI files (and intermediate tarballs) into `results.tar.gz`

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment. `fasterq-dump` needs outbound **IPv4** internet access on the workers (SRA data
is served from AWS S3, which has no IPv6 address).

## 1. Prepare Sample Input Data

The repository ships ready-made inputs in `examples/{1,10,1000}/`: an SRA ID list (`sra_ids.txt`) and the
crAssphage reference (`crassphage.fna`). This notebook uses the single-run set, `examples/1/`. Nothing needs
to be downloaded up front — the `fasterq-dump` jobs fetch the reads at run time.

In [ ]:
!cat examples/1/sra_ids.txt

## 2. Build the Apptainer Container

All jobs run inside one Apptainer container (`container/sra.def`) bundling SRA Toolkit, Bowtie2 and
Samtools. Building it can take a few minutes; skip the rebuild if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f container/sra.sif ]; then
    (cd container && apptainer build sra.sif sra.def)
else
    echo "Container already built: container/sra.sif"
fi

## 3. Generate the Workflow

The following cell uses the `SraSearchWorkflow` class from `sra_search.py` — the same generator used by the
command-line tool (`./sra-search.py`) — to build the Pegasus workflow DAG (site, transformation, and replica
catalogs) and writes it to `workflow.yml`. Nothing about the workflow is defined here: change the pipeline in
`sra_search.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from sra_search import SraSearchWorkflow, read_sra_ids

SRA_ID_LIST = "examples/1/sra_ids.txt"
REFERENCE = "examples/1/crassphage.fna"
DAGFILE = "workflow.yml"
EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

print(f"SRA IDs: {read_sra_ids(SRA_ID_LIST)}")

workflow = SraSearchWorkflow(SRA_ID_LIST, REFERENCE, dagfile=DAGFILE)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nSRA search workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. The graph shows the single
`bowtie2-build` index job feeding one `fasterq-dump` → `bowtie2` chain per SRA ID, all converging on the
merge job(s).

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog
chosen above.

The CLI version of this generator (`./sra-search.py`) writes the same workflow non-interactively and prints the
`pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages one final output to `output/`:

| Output | Description |
|--------|-------------|
| `results.tar.gz` | Sorted BAM + BAI file for every SRA ID, aligned against the reference |

In [ ]:
!ls -l output/

List the aligned BAM/BAI files in the tarball:

In [ ]:
!tar -tzvf output/results.tar.gz